# Validation — Dokumacı Fig 5.15: tuned through-flow expansion chamber

Reference: E. Dokumacı, *Duct Acoustics: Fundamentals and Applications to Mufflers and Silencers*, Cambridge University Press (2021), §5.4.5, Fig 5.15.

This notebook validates the Nefes acoustic-network layer against a classic muffler transmission-loss (TL) case.
A pure through-flow expansion chamber, and the same chamber tuned by extending its inlet / outlet ducts into the chamber to form closed quarter-wave side branches.
The geometry and gas state are fully specified in the figure caption (no digitising required).
Dokumacı gives a closed-form TL (the unified formula Eq 5.89), so we validate against an *exact analytic reference*.

**Geometry & gas (Fig 5.15 caption)**

| quantity | symbol | value |
|---|---|---|
| inlet / outlet area | $S_1$ | $31.67\ \mathrm{cm^2}$ |
| chamber area | $S_c$ | $471.35\ \mathrm{cm^2}$ |
| chamber length | $L$ | $71\ \mathrm{cm}$ |
| inlet extension | $L_A$ | $35.5\ \mathrm{cm}=L/2$ |
| outlet extension | $L_B$ | $17.75\ \mathrm{cm}=L/4$ |
| sound speed | $c_o$ | $344.9\ \mathrm{m/s}$ |
| density | $\rho_o$ | $1.1935\ \mathrm{kg/m^3}$ |
| Mach (inlet/outlet duct) | $\overline{M}_o$ | $0,\ 0.1,\ 0.2$ |

**Plan**
1. **Part 1** — pure chamber (Fig 5.15a) as an Nefes network; TL vs frequency at $\overline{M}_o=0,0.1,0.2$, against the analytic formula. *(headline quantitative check)*
2. **Part 2** — reproduce all three panels (a, b, c) from Dokumacı's unified Eq (5.89).
3. **Part 3** — build the tuned chambers (b, c) as branched Nefes networks with closed quarter-wave stubs, and compare to Eq (5.89).


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.thermo.configure import perfect_gas
from nefes.perturbation import perturbation_response
from nefes.plotting import use_nefes_theme

_ = use_nefes_theme()

#  --- gas (calorically perfect air, tuned so that c == c_o of the caption) ---
GAMMA, R = 1.4, 287.0
GAS = perfect_gas(R, GAMMA)

#  --- Fig 5.15 geometry ---
S1 = 31.67e-4    # inlet / outlet cross-sectional area [m^2]
SC = 471.35e-4   # chamber cross-sectional area [m^2]
L = 0.71         # chamber length [m]
LA = 0.355       # inlet-duct extension (= L/2) [m]
LB = 0.1775      # outlet-duct extension (= L/4) [m]
CO = 344.9       # sound speed [m/s]
P0 = 101325.0    # static pressure [Pa]
T0 = CO ** 2 / (GAMMA * R)        # temperature giving c == CO
RHO0 = P0 / (R * T0)
SANN = SC - S1   # annular dead-space area around an extended duct
M_RATIO = SC / S1  # area ratio m

FREQS = np.linspace(1.0, 1000.0, 1000)  # analysis sweep [Hz]
print(f"area ratio m = {M_RATIO:.3f}")
print(f"first chamber critical f1 = c/2L = {CO / (2 * L):.1f} Hz")
print(f"stub tuning  c/4LA = {CO / (4 * LA):.1f} Hz,  c/4LB = {CO / (4 * LB):.1f} Hz")
print(f"T0 = {T0:.2f} K, rho0 = {RHO0:.4f} kg/m^3 (caption: 1.1935)")


## Part 1 — pure expansion chamber (Fig 5.15a)

We build the literal geometry with the `Network` builder — add the elements, connect them with directed edges, then `solve()`:

```
inlet ──[S1]── expansion ──[Sc]── duct(L) ──[Sc]── contraction ──[S1]── outlet
```

Transmission loss can be obtained from the transmission coefficient read from the acoustic **scattering matrix** between the inlet-pipe edge and the outlet-pipe edge.

$$\mathrm{TL} = -20\log_{10}|\tau|.$$

The analytic reference is Eq (5.89):

$$\mathrm{TL} = 10\log_{10}\!\Big[1 + \tfrac14\big(m-\tfrac1m\big)^2\sin^2(k L)\Big], \qquad k=\frac{2\pi f}{c_o},\ m=\frac{S_c}{S_1}.$$


In [ ]:
def build_pure_chamber(mach):
    """Pure through-flow expansion chamber at a given inlet/outlet-duct Mach number.

    Returns (Solution, inlet_edge, outlet_edge); the edge ids are captured straight
    from connect() so the TL reader needs no edge bookkeeping.
    """
    mdot = RHO0 * (mach * CO) * S1                   # 0 when quiescent
    net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=max(mdot, 0.05))
    inlet = net.add(cat.mass_flow_inlet(mdot, T0))
    expansion = net.add(cat.isentropic_area_change("expansion"))    # S1 -> Sc
    chamber = net.add(cat.duct(L))
    contraction = net.add(cat.isentropic_area_change("contraction"))  # Sc -> S1
    outlet = net.add(cat.pressure_outlet(P0, T0))
    e_in = net.connect(inlet, expansion, S1)         # inlet pipe
    net.connect(expansion, chamber, SC)
    net.connect(chamber, contraction, SC)
    e_out = net.connect(contraction, outlet, S1)     # outlet pipe
    sol = net.solve()
    assert sol.converged, "mean-flow solve did not converge"
    return sol, e_in, e_out


def tl_two_port(sol, freqs, edge_in, edge_out):
    """TL of an unbranched 2-port from the acoustic scattering matrix (equal pipes)."""
    resp = perturbation_response(sol.problem, sol.x, freqs)
    S = resp.acoustic_scattering_matrix(edge_in, edge_out)
    tau = S[:, 1, 0]                                 # outgoing@outlet per incoming@inlet
    return -20.0 * np.log10(np.abs(tau))


def tl_pure_analytic(freqs):
    """Textbook expansion-chamber TL (= Eq 5.89 with zeta5 = zeta6 = inf)."""
    k = 2.0 * np.pi * freqs / CO
    return 10.0 * np.log10(1.0 + 0.25 * (M_RATIO - 1.0 / M_RATIO) ** 2 * np.sin(k * L) ** 2)


In [ ]:
tl_a = {}
for mach in (0.0, 0.1, 0.2):
    sol, e_in, e_out = build_pure_chamber(mach)
    m_pipe = abs(float(sol.field("M")[e_in]))
    tl_a[mach] = (m_pipe, tl_two_port(sol, FREQS, e_in, e_out))

tl_a_ref = tl_pure_analytic(FREQS)
for mach, (m_pipe, tl) in tl_a.items():
    print(f"M_in = {m_pipe:.4f}   max|Nefes - analytic(no flow)| = {np.max(np.abs(tl - tl_a_ref)):.3f} dB")


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQS, y=tl_a_ref, name="Eq (5.89), analytic",
                         line=dict(color="black", width=4), opacity=0.35))
colors = {0.0: "#1f77b4", 0.1: "#ff7f0e", 0.2: "#2ca02c"}
for mach, (m_pipe, tl) in tl_a.items():
    fig.add_trace(go.Scatter(x=FREQS, y=tl, name=f"Nefes, M̄ₒ = {mach}",
                             line=dict(color=colors[mach], width=1.6)))
fig.update_layout(
    title="Fig 5.15a — pure expansion chamber: Nefes vs analytic",
    xaxis_title="Frequency, Hz", yaxis_title="Transmission Loss, dB",
    width=820, height=460,
    legend=dict(x=0.01, y=0.99))
fig.update_yaxes(range=[0, 25])
fig


**Result.** At $\overline{M}_o=0$ the Nefes network reproduces the analytic curve to machine precision; the lobe peaks at $\mathrm{TL}_{\max}=10\log_{10}[1+\tfrac14(m-1/m)^2]\approx 17.5\ \mathrm{dB}$ and the nulls fall at $f=n\,c_o/2L$.
The mean-flow curves ($\overline{M}_o=0.1,0.2$) sit within a few hundredths of a dB of the no-flow analytic, confirming Dokumacı's remark that for the pure chamber the mean-flow effect is *barely discernible*.


## Part 2 — Dokumacı's unified formula, Eq (5.89)

For this chamber family the inverse FRF inside the modulus brackets of Eq (5.89) is

$$\frac{p_1^+}{p_2^+}=\cos(k L)-\tfrac{\mathrm i}{2}\Big(m+\tfrac1m\Big)\sin(kL)
-\frac{\mathrm i(B_1+m)}{2A_2B_2\zeta_5\zeta_6}\Big(A_1+\tfrac1m\Big)\sin(kL)
+\frac{1}{2A_2\zeta_5}\!\big[-(1+A_1 m)\cos kL+\mathrm i(A_1+\tfrac1m)\sin kL\big]
+\frac{1}{2B_2\zeta_6}\!\big[-(B_1+m)\cos kL+\mathrm i(1+\tfrac{B_1}{m})\sin kL\big],$$

with $\mathrm{TL}=20\log_{10}|p_1^+/p_2^+|$.
For the through-flow chamber (Table 5.2, row a) $A_1=-1,\,A_2=1,\,B_1=-1,\,B_2=-1$.
A rigid closed side branch of length $\ell$ contributes $\zeta=1/(\mathrm i\tan k\ell)$; an absent branch is $\zeta\to\infty$.

- panel (a): $\zeta_5=\zeta_6=\infty$
- panel (b): $\zeta_5=1/(\mathrm i\tan kL_A)$ (inlet tuned to $f_1$), $\zeta_6=\infty$
- panel (c): additionally $\zeta_6=1/(\mathrm i\tan kL_B)$ (outlet tuned to $f_2$)


In [ ]:
def tl_eq589(freqs, zeta5, zeta6, A1=-1.0, A2=1.0, B1=-1.0, B2=-1.0):
    """Dokumacı's unified expansion-chamber TL, Eq (5.89)."""
    k = 2.0 * np.pi * freqs / CO
    kl = k * L
    c, s = np.cos(kl), np.sin(kl)
    z5, z6 = zeta5(freqs), zeta6(freqs)
    inv = (c - 0.5j * (M_RATIO + 1.0 / M_RATIO) * s
           - 1j * (B1 + M_RATIO) / (2 * A2 * B2 * z5 * z6) * (A1 + 1.0 / M_RATIO) * s
           + 1.0 / (2 * A2 * z5) * (-(1 + A1 * M_RATIO) * c + 1j * (A1 + 1.0 / M_RATIO) * s)
           + 1.0 / (2 * B2 * z6) * (-(B1 + M_RATIO) * c + 1j * (1 + B1 / M_RATIO) * s))
    return 20.0 * np.log10(np.abs(inv))


INF = lambda f: np.full_like(f, 1e12, dtype=complex)             # absent side branch
closed_stub = lambda length: (lambda f: 1.0 / (1j * np.tan(2.0 * np.pi * f / CO * length)))

tl_b_ref = tl_eq589(FREQS, closed_stub(LA), INF)                 # inlet tuned
tl_c_ref = tl_eq589(FREQS, closed_stub(LA), closed_stub(LB))     # inlet + outlet tuned


In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("(a) pure chamber",
                                    "(b) inlet duct tuned (LA = L/2)",
                                    "(c) inlet + outlet ducts tuned"))
for row, tl in enumerate((tl_a_ref, tl_b_ref, tl_c_ref), start=1):
    fig.add_trace(go.Scatter(x=FREQS, y=tl, line=dict(color="black", width=1.5),
                             showlegend=False), row=row, col=1)
fig.update_yaxes(title_text="TL, dB", range=[0, 45])
fig.update_xaxes(title_text="Frequency, Hz", row=3, col=1)
fig.update_layout(title="Reproduction of Fig 5.15 from Eq (5.89)",
                  width=820, height=760)
fig


These three curves reproduce Fig 5.15(a–c): the tuned cases turn the chamber's TL nulls at $f_1=c_o/2L$ (panel b) and additionally at $f_2=2c_o/2L$ (panel c) into sharp resonance peaks at $f=n c_o/4L_A$ and $n c_o/4L_B$.


## Part 3 — tuned chambers as branched Nefes networks

The extended inlet/outlet ducts are modelled *geometrically*: the extended duct ($S_1$) runs into the chamber, and the annular dead space around it ($S_c-S_1$), capped by the chamber end wall, is a **closed quarter-wave stub**.

```
inlet ─[S1]─ ext.duct(LA) ─┬─[Sc]─ chamber ─ … ─ contraction ─[S1]─ outlet
                           └─[Sann]─ stub(LA) ─ wall   (closed dead leg)
```

A scattering measurement neutralises *every* boundary terminal to anechoic, which would silently open the stub-end walls and kill the quarter-wave resonance.
So we keep them rigid with `perturbation_response(..., freeze=[wall, …])`: each closed stub is folded into the operator at $R=+1$, the interior wall ports drop out, and the device reduces to a genuine inlet→outlet **two-port** — read out exactly like the unbranched chamber in Part 1.


In [ ]:
def build_extended(la, lb=None):
    """Tuned expansion chamber with extended inlet (and optionally outlet) ducts.

    Returns (Solution, ports) where ports gives the inlet/outlet edges (captured straight
    from connect()) and the stub-end wall nodes to freeze (kept rigid during the acoustic
    measurement).  Quiescent mean flow (TL is mean-flow insensitive here for the regime of
    interest, as Part 1 showed).
    """
    net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=0.05)
    if lb is None:                                   # case (b): inlet extension only
        inlet = net.add(cat.mass_flow_inlet(0.0, T0))
        ext_in = net.add(cat.duct(la))               # extended inlet pipe (S1)
        tip = net.add(cat.junction())                # tip of the extension
        chamber = net.add(cat.duct(L - la))          # chamber remainder (Sc)
        contraction = net.add(cat.isentropic_area_change("contraction"))
        outlet = net.add(cat.pressure_outlet(P0, T0))
        stub = net.add(cat.duct(la))                 # annular stub (Sann)
        wall = net.add(cat.wall())                   # stub end wall
        e_in = net.connect(inlet, ext_in, S1)
        net.connect(ext_in, tip, S1)
        net.connect(tip, chamber, SC)
        net.connect(chamber, contraction, SC)
        e_out = net.connect(contraction, outlet, S1)
        net.connect(tip, stub, SANN)
        net.connect(stub, wall, SANN)
        ports = dict(in_edge=e_in, out_edge=e_out, walls=[wall])
    else:                                            # case (c): both ducts extended
        inlet = net.add(cat.mass_flow_inlet(0.0, T0))
        ext_in = net.add(cat.duct(la))               # extended inlet pipe
        tip_in = net.add(cat.junction())             # inlet-extension tip
        mid = net.add(cat.duct(L - la - lb))         # mid chamber (Sc)
        tip_out = net.add(cat.junction())            # outlet-extension tip
        ext_out = net.add(cat.duct(lb))              # extended outlet pipe
        outlet = net.add(cat.pressure_outlet(P0, T0))
        stub_in = net.add(cat.duct(la))              # inlet annular stub
        wall_in = net.add(cat.wall())
        stub_out = net.add(cat.duct(lb))             # outlet annular stub
        wall_out = net.add(cat.wall())
        e_in = net.connect(inlet, ext_in, S1)
        net.connect(ext_in, tip_in, S1)
        net.connect(tip_in, mid, SC)
        net.connect(mid, tip_out, SC)
        net.connect(tip_out, ext_out, S1)
        e_out = net.connect(ext_out, outlet, S1)
        net.connect(tip_in, stub_in, SANN)
        net.connect(stub_in, wall_in, SANN)
        net.connect(tip_out, stub_out, SANN)
        net.connect(stub_out, wall_out, SANN)
        ports = dict(in_edge=e_in, out_edge=e_out, walls=[wall_in, wall_out])
    sol = net.solve()
    assert sol.converged, "mean-flow solve did not converge"
    return sol, ports


def tl_branched(sol, freqs, ports):
    """TL of a branched muffler, read as a clean 2-port by freezing the stub-end walls.

    Freezing keeps each wall rigid (R = +1) during the measurement, so the closed stubs
    are folded into the operator and only the inlet/outlet remain as ports — the inlet->outlet
    scattering then reads out exactly like the unbranched chamber (Part 1), with no multiport
    condensation.
    """
    resp = perturbation_response(sol.problem, sol.x, freqs, freeze=ports["walls"])
    S = resp.acoustic_scattering_matrix(ports["in_edge"], ports["out_edge"])
    tau = S[:, 1, 0]                                 # outgoing@outlet per incoming@inlet
    return -20.0 * np.log10(np.abs(tau))


In [ ]:
sol_b, ports_b = build_extended(LA)
sol_c, ports_c = build_extended(LA, LB)
tl_b_fns = tl_branched(sol_b, FREQS, ports_b)
tl_c_fns = tl_branched(sol_c, FREQS, ports_c)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("(b) inlet duct tuned", "(c) inlet + outlet ducts tuned"))
for row, (tl_fns, tl_ref) in enumerate([(tl_b_fns, tl_b_ref), (tl_c_fns, tl_c_ref)], start=1):
    fig.add_trace(go.Scatter(x=FREQS, y=tl_ref, name="Eq (5.89), lumped side branch",
                             line=dict(color="black", width=4), opacity=0.3,
                             showlegend=(row == 1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=FREQS, y=tl_fns, name="Nefes, geometric closed stub",
                             line=dict(color="#d62728", width=1.6),
                             showlegend=(row == 1)), row=row, col=1)
for f_tune in (CO / (4 * LA), CO / (4 * LB), 3 * CO / (4 * LA)):
    fig.add_vline(x=f_tune, line=dict(color="grey", dash="dot", width=1))
fig.update_yaxes(title_text="TL, dB", range=[0, 60])
fig.update_xaxes(title_text="Frequency, Hz", row=2, col=1)
fig.update_layout(title="Tuned chambers — Nefes branched network vs Eq (5.89)",
                  width=860, height=620, template="plotly_white",
                  legend=dict(x=0.01, y=0.99))
fig


**Result & interpretation.**
The Nefes branched networks predict the tuning resonances at exactly $c_o/4L_A$ (243 Hz), its third harmonic $3c_o/4L_A$ (729 Hz), and $c_o/4L_B$ (486 Hz), and they keep the chamber nulls — the *acoustically robust* features that define the device.

The off-resonance envelope differs somewhat from Eq (5.89).
This is a genuine **modelling** difference, not a solver error: Dokumacı's $\zeta_5,\zeta_6$ are *lumped* side-branch impedances coupled at the chamber face and normalised to the branch's own $\rho c/S$, whereas the Nefes network couples a *finite-area* annular stub through quasi-static junction conservation at the duct tip.
Both are legitimate one-dimensional idealisations of the same three-dimensional extended-duct chamber; the true answer (and the measured one) lies between them.
Part 1 — where no such idealisation is involved — is the clean machine-precision check.

## Export for Nemo

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in Nemo.

In [ ]:
import os, tempfile

network, sol = sol_b.network, sol_b  # the primary Network and its mean-flow Solution
_out = os.path.join(tempfile.mkdtemp(), "dokumaci_expansion_chamber.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)